# 01. Planner Agent (`ResearchPlanner`) Demo

### Architecture Role & Responsibilities
The **Planner Agent** serves as the Lead Investment Strategist and Query Dispatcher. It sits at the workflow entry point and evaluates user queries under two distinct operational flows:

1. **Flow 1 (Research Required)**: Emits minimal structured JSON `{"research_required": true, "ticker": "<TICKER>"}` for new company inquiries, dispatching to the downstream Router / Orchestrator which dynamically selects specialists.
2. **Flow 2 (Direct In-Memory Resolution)**: Answers directly from memory or internal knowledge for general financial concepts, greetings, and session history recall—short-circuiting the graph directly to `END`.

In [1]:
import sys
from pathlib import Path

# Resolve src directory
project_root = Path.cwd().parent if Path.cwd().name == 'demo' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from research_agent.agents.planner import planner_agent
from research_agent.agent_framework import GraphMemory
from research_agent.interfaces import get_llm

# Initialize Planner Agent (lookup_ticker is built-in; memory tools injected from session)
memory = GraphMemory()
llm = get_llm()
planner = planner_agent(llm=llm, tools=memory.as_tools())
print(f"Planner Agent initialized: {planner.name}")
print(f"Registered tools: {list(planner.tools.keys())}")

Planner Agent initialized: ResearchPlanner
Registered tools: ['lookup_ticker', 'get_history_states', 'get_state', 'list_state_ids', 'list_threads']


## Scenario 1: Live Company Research Query (Flow 1: Minimal JSON Output)
When given a new company inquiry, the Planner retrieves the verified ticker symbol from Yahoo Finance API via `lookup_ticker` rather than relying on pre-trained LLM memory, and outputs a clean JSON dispatch directive (`{"research_required": true, "ticker": "NVDA"}`).

In [2]:
# Scenario 1: Live Company Research Query (Flow 1: Minimal JSON Output)
query_1 = "Analyse NVIDIA (NVDA): evaluate Q2 datacenter growth, Blackwell ramp, and margin sustainability."
print(f"Sending Input: '{query_1}'\n")

resp_1 = planner.run(query_1)
print(f"Response Type : {resp_1.response_type}")
print(f"Is Dict       : {isinstance(resp_1.content, dict)}")
for step in resp_1.trace:
    if step.step_type == "tool_call":
        print(f"TOOL CALL     : {step.content.get('tool')}({step.content.get('arguments')})")
    elif step.step_type == "tool_result":
        print(f"TOOL RESULT   : {step.content}")
print("\n--- PLANNER OUTPUT (JSON) ---")
print(resp_1)

# Persist the completed research run into graph memory (simulating the completed workflow execution)
from research_agent.agent_framework import GraphState

completed_nvda_state = GraphState(
    question=query_1,
    output="NVDA Q2 Revenue: $30.04B, Gross Margin: 75.1%, Operating Income: $18.64B, Blackwell ramp on track.",
    agents={"EarningsAnalyst": "Q2 revenue $30.04B with 75.1% gross margin."},
    tools={"get_earnings_data": {"revenue_b": 30.04, "gross_margin_pct": 75.1}},
)
saved_id = memory.write(completed_nvda_state)
print(f"\n[Workflow Completed] Persisted research state into GraphMemory (ID: {saved_id})")

Sending Input: 'Analyse NVIDIA (NVDA): evaluate Q2 datacenter growth, Blackwell ramp, and margin sustainability.'

Response Type : json
Is Dict       : True
TOOL CALL     : lookup_ticker({'company_name': 'NVIDIA', 'query': 'NVIDIA'})
TOOL RESULT   : {
  "ticker": "NVDA",
  "company": "NVIDIA Corporation",
  "source": "yahoo_finance"
}

--- PLANNER OUTPUT (JSON) ---
{
  "research_required": true,
  "ticker": "NVDA"
}

[Workflow Completed] Persisted research state into GraphMemory (ID: state_20261012_225023_3bee7d)


## Scenario 2: Financial Concept Explanation (Flow 2: Plain Text Output)
When asked an educational or conceptual financial question, the Planner directly answers in plain text without JSON, setting `response_type = 'text'` and short-circuiting to `END`.

In [3]:
# Scenario 2: Financial Concept Explanation (Flow 2: Plain Text Output)
query_2 = "What is operating margin and why is it important in tech equities?"
print(f"Sending Input: '{query_2}'\n")

resp_2 = planner.run(query_2)
print(f"Response Type : {resp_2.response_type}")
print(f"Is String     : {isinstance(resp_2.content, str)}")
print("\n--- PLANNER OUTPUT (PLAIN TEXT) ---")
print(resp_2.content)

Sending Input: 'What is operating margin and why is it important in tech equities?'

Response Type : text
Is String     : True

--- PLANNER OUTPUT (PLAIN TEXT) ---
Operating margin is a profitability metric that measures the proportion of a company’s revenue that remains after covering all operating expenses, but before interest and taxes are taken into account. It is calculated as:

\[
\text{Operating Margin} = \frac{\text{Operating Income (EBIT)}}{\text{Revenue}} \times 100\%
\]

**Why it matters for tech equities:**

| Reason | Explanation |
|--------|-------------|
| **Indicator of Core Business Efficiency** | In technology firms, a large portion of costs are tied to R&D, product engineering, and high‑speed infrastructure. A strong operating margin shows that the core business (e.g., semiconductor design, cloud services, software licensing) is generating solid profit from its primary activities. |
| **Scalability & Leverage** | Many tech companies have high fixed costs (e.g., data 

## Scenario 3: Session History & Timeline Recall (Flow 2)
When the user asks meta questions about prior session research, the Planner inspects memory and reports factually.

In [4]:
# Scenario 3: Session History & Timeline Recall (Flow 2)
query_3 = "On what companies did I do research till now in this session?"
print(f"Sending Input: '{query_3}'\n")

resp_3 = planner.run(query_3)
print(f"Response Type : {resp_3.response_type}")
print("\n--- PLANNER OUTPUT ---")
print(resp_3.content)

Sending Input: 'On what companies did I do research till now in this session?'

Response Type : text

--- PLANNER OUTPUT ---
In this session you have only researched one company so far:

- **NVIDIA Corp. (NVDA)**

No other tickers or companies have been analyzed yet. Let me know which additional stocks you’d like to examine.


## Scenario 4: Direct Memory Inspection Tools Access
The Planner is equipped with `GraphMemory` tools (`get_history_states`, `get_state`, `list_state_ids`, `list_threads`). Here we demonstrate programmatic interaction with the memory store: persisting a completed research state, listing state IDs, and retrieving paginated history.

In [5]:
# Scenario 4: Direct Memory Inspection Tools Access
# 1. Inspect memory via list_state_ids tool
print("--- 1. memory.list_state_ids() ---")
print(memory.list_state_ids())

# 2. Inspect paginated memory via get_history_states tool
print("\n--- 2. memory.get_history_states(page=1, limit=5) ---")
print(memory.get_history_states(page=1, limit=5))

# 3. Retrieve exact state via get_state tool
print(f"\n--- 3. memory.get_state('{saved_id}') ---")
print(memory.get_state(saved_id))

# 4. List conversation threads via list_threads tool
print("\n--- 4. memory.list_threads() ---")
print(memory.list_threads())

--- 1. memory.list_state_ids() ---
[
  "state_20261012_225023_3bee7d"
]

--- 2. memory.get_history_states(page=1, limit=5) ---
{
  "page": 1,
  "limit": 5,
  "total_records": 1,
  "total_pages": 1,
  "has_next_page": false,
  "has_prev_page": false,
  "thread_id": null,
  "records": [
    {
      "question": "Analyse NVIDIA (NVDA): evaluate Q2 datacenter growth, Blackwell ramp, and margin sustainability.",
      "output": "NVDA Q2 Revenue: $30.04B, Gross Margin: 75.1%, Operating Income: $18.64B, Blackwell ramp on track.",
      "state_unique_id": "state_20261012_225023_3bee7d",
      "thread_id": "default_session",
      "agents": {
        "EarningsAnalyst": "Q2 revenue $30.04B with 75.1% gross margin."
      },
      "tools": {
        "get_earnings_data": {
          "revenue_b": 30.04,
          "gross_margin_pct": 75.1
        }
      },
      "trajectory": [],
      "refine_count": 0,
      "review": {},
      "timestamp": "2026-10-12 22:50:23",
      "input": "Analyse NVIDIA (NV

## Scenario 5: Planner Agent Autonomous Memory Retrieval
When the user asks a follow-up inquiry referencing prior research, the Planner uses its registered memory tools (`get_history_states`, `get_state`) to fetch the historical state, avoiding duplicate research cycles.

In [6]:
# Scenario 5: Planner Agent Autonomous Memory Retrieval
query_4 = "What were the financial figures from my previous NVDA research in memory?"
print(f"Sending Input: '{query_4}'\n")

resp_4 = planner.run(query_4)
print(f"Response Type : {resp_4.response_type}")
print("\n--- PLANNER OUTPUT (RECALLED FROM MEMORY) ---")
print(resp_4.content)

print("\n--- EXECUTION TRACE ---")
for step in resp_4.trace:
    if step.step_type == "tool_call":
        print(f"TOOL CALL   : {step.content.get('tool')}({step.content.get('arguments')})")
    elif step.step_type == "final_answer":
        print(f"FINAL ANSWER: Generated ({len(str(step.content))} characters)")

Sending Input: 'What were the financial figures from my previous NVDA research in memory?'

Response Type : text

--- PLANNER OUTPUT (RECALLED FROM MEMORY) ---
Here are the key financial figures that were captured from your earlier NVIDIA (NVDA) research:

| Metric (Q2 2024) | Value |
|------------------|-------|
| **Revenue** | **$30.04 billion** |
| **Gross Margin** | **75.1 %** |
| **Operating Income** | **$18.64 billion** |
| **Blackwell (GPU) ramp status** | **On track** – the transition to the Blackwell architecture was noted as proceeding as expected. |

These numbers came from the Q2 earnings data that was retrieved during the prior analysis. Let me know if you’d like a deeper dive into any of these items (e.g., datacenter segment growth, margin sustainability, or forward‑looking guidance).

--- EXECUTION TRACE ---
FINAL ANSWER: Generated (649 characters)
